In [1]:
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
     AutoTokenizer,
     AutoModelForTokenClassification,
     TrainingArguments,
     Trainer,
     DataCollatorForTokenClassification
 )
from sklearn.model_selection import train_test_split
import os

PROCESSED_DIR = "../data/processed/"
MODEL_DIR = "../models/"
os.makedirs(MODEL_DIR, exist_ok = True)

# Load BIO labeled data
with open(os.path.join(PROCESSED_DIR, "bio_labeled_sample.json")) as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
print(f"Sample words:  {records[0]['words'][:60]}")
print(f"Sample words:  {records[0]['labels'][:60]}")

Loaded 30000 records
Sample words:  ['Worked', 'fine', 'for', '2', 'days', 'burning', 'and', 'itching', 'returned', 'on', 'day', '3']
Sample words:  ['O', 'O', 'O', 'O', 'O', 'B-ADR', 'O', 'O', 'O', 'O', 'O', 'O']


In [2]:
# import re

# def clean_words(words):
#     return [re.sub(r'^[^a-zA-Z0-9]+|[^a-zA-Z0-9]+$', '', w) for w in words]

# records[0]['words'] = clean_words(records[0]['words'])

# print("Cleaned words:", records[0]['words'][:60])
# print("Labels:", records[0]['labels'][:60])


In [3]:
# Every NER model needs a consistent label → integer mapping
# -100 is reserved by PyTorch for ignored tokens — do NOT use it as a label ID
LABEL2ID = {"O": 0, "B-ADR": 1, "I-ADR": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("Label Mappings", LABEL2ID)

Label Mappings {'O': 0, 'B-ADR': 1, 'I-ADR': 2}


In [4]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded")
print("Vocab size: ", tokenizer.vocab_size)

test = tokenizer.tokenize("hepatotoxicity nausea")
print("Tokenized: ", test)

Tokenizer Loaded
Vocab size:  28996
Tokenized:  ['he', '##pa', '##to', '##to', '##xi', '##city', 'nausea']


In [5]:
def tokenize_and_align(words, labels, tokenizer, max_len= 128):
    """
    Takes word-level tokens and BIO labels.
    Returns input_ids, attention_mask, and aligned label_ids.
    
    Key rule:
        - First subword of a word → gets the real label
        - Continuation subwords (##...) → get -100 (ignored in loss)
        - [CLS] and [SEP] special tokens → get -100
    """
    tokenized = tokenizer(
        words,
        is_split_into_words = True, # tells tokenizer input is already split
        truncation = True,
        max_length = max_len,
        padding = "max_length"
    )

    word_ids = tokenized.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
       
        if word_id is None:
            # Special tokens [CLS], [SEP], [PAD] → ignore
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            # First subword of a new word → assign real label
            aligned_labels.append(LABEL2ID[labels[word_id]])
        else:
            # Continuation subword → ignore
            aligned_labels.append(-100)
        prev_word_id= word_id
       
    return {
         "input_ids": tokenized["input_ids"],
         "attention_mask": tokenized["attention_mask"],
         "labels":         aligned_labels
    }

In [6]:
# test on one record
sample = records[0]
test = tokenize_and_align(sample["words"], sample["labels"], tokenizer)

print("input_ids length", len(test["input_ids"]))
print("labels length", len(test["labels"]))
print("\nFirst 20 token alignments:")
print(f"{'Token':<20} {'Label ID':<10} {'Meaning'}")
print("-" * 45)
for i in range(20):
    token = tokenizer.convert_ids_to_tokens(test["input_ids"])[i]
    label_id = test["labels"][i]
    meaning =ID2LABEL.get(label_id, "ignore") if label_id != -100 else "-100 (ignored)"
    print(f"{token:<20} {str(label_id):<10} {meaning}")
    
        


input_ids length 128
labels length 128

First 20 token alignments:
Token                Label ID   Meaning
---------------------------------------------
[CLS]                -100       -100 (ignored)
worked               0          O
fine                 0          O
for                  0          O
2                    0          O
days                 0          O
burning              1          B-ADR
and                  0          O
it                   0          O
##ching              -100       -100 (ignored)
returned             0          O
on                   0          O
day                  0          O
3                    0          O
[SEP]                -100       -100 (ignored)
[PAD]                -100       -100 (ignored)
[PAD]                -100       -100 (ignored)
[PAD]                -100       -100 (ignored)
[PAD]                -100       -100 (ignored)
[PAD]                -100       -100 (ignored)


In [7]:
class NERDataset(Dataset):

    def __init__(self, records, tokenizer, max_len = 128):
        self.samples = [ 
            tokenize_and_align(r["words"], r["labels"], tokenizer)
            for r in records
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.samples[idx]["input_ids"]),
            "attention_mask": torch.tensor(self.samples[idx]["attention_mask"]),
            "labels": torch.tensor(self.samples[idx]["labels"])
        }

# Train / val / test split — stratify not possible for NER so just random
train_records, temp_records = train_test_split(records, test_size = 0.3, random_state = 42)
val_records, test_records = train_test_split(temp_records, test_size = 0.5, random_state = 42)

print("Train len: ", len(train_records), "| Val len: ", len(val_records), "| Test records: ", len(test_records))

print("Building datasets (this takes ~2 mins)...")
train_ds = NERDataset(train_records, tokenizer)
val_ds = NERDataset(val_records, tokenizer)
test_ds = NERDataset(test_records, tokenizer)

Train len:  21000 | Val len:  4500 | Test records:  4500
Building datasets (this takes ~2 mins)...


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels = len(LABEL2ID), # 3: O, B-ADR, I-ADR
    id2label= ID2LABEL,
    label2id = LABEL2ID
)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

Total parameters:     107,721,987
Trainable parameters: 107,721,987


In [9]:
from seqeval.metrics import f1_score, classification_report as seq_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis =2) # shape: (batch, seq_len)

    true_labels, true_preds = [], []

    for pred_seq, label_seq in zip(predictions, labels):
        true_label_seq, true_pred_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l!= -100: # skip ignored
                true_label_seq.append(ID2LABEL[l])
                true_pred_seq.append(ID2LABEL[p])
        true_labels.append(true_label_seq)
        true_preds.append(true_pred_seq)

    return {
        "f1": f1_score(true_labels, true_preds)
    }

In [10]:
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, "biobert_ner"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,            # standard for BERT fine-tuning
    warmup_ratio=0.1,              # 10% of steps used for LR warmup
    weight_decay=0.01,             # L2 regularization
    eval_strategy="epoch",   # evaluate after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,   # auto-loads best checkpoint
    logging_steps=50,
    fp16=torch.cuda.is_available() # mixed precision if GPU available
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset= val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
    
)

print("Trainer ready")
print(f"Steps per epoch: {len(train_ds) // 16}")
print(f"Total training steps: {(len(train_ds) // 16) * 3}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready
Steps per epoch: 1312
Total training steps: 3936


In [ ]:
print("Starting fine-tuning...")
trainer.train()

Starting fine-tuning...


C:\Users\HP\Desktop\drug_safety_insights_nlp\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [11]:
model_path = "../models/biobert_ner/checkpoint-3939"

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
trainer = Trainer(
    model = model,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [13]:
# Evaluate on test set
print("Evaluating on test set...")
predictions, labels, _ = trainer.predict(test_ds)

Evaluating on test set...


C:\Users\HP\Desktop\drug_safety_insights_nlp\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [17]:
# Convert logits to predicted label IDs
pred_ids = np.argmax(predictions, axis =2) 
true_labels, true_preds = [], []

for pred_seq, label_seq in zip(pred_ids, labels):
    true_pred_seq, true_label_seq = [],[]
    for p, l in zip(pred_seq, label_seq):
        if l!= -100:
            true_pred_seq.append(ID2LABEL[p])
            true_label_seq.append(ID2LABEL[l])
    true_preds.append(true_pred_seq)
    true_labels.append(true_label_seq)

# detailed report
print(seq_report(true_labels, true_preds))

              precision    recall  f1-score   support

         ADR       0.73      0.80      0.76     10292

   micro avg       0.73      0.80      0.76     10292
   macro avg       0.73      0.80      0.76     10292
weighted avg       0.73      0.80      0.76     10292

